# model_09 — deneme 2 · DİL EĞİTİMİNE GEÇİŞ

Önceden kayıt: `belge/onkayit/model_11.md`.

Kullanıcı, 18 Eylül: *"biz artık r1 r2 e1 gibi şeyleri ölçmüyoruz. biz
artık dil eğitimine geçtik. bir dil eğitimi pratiği gibi olmalı."* ve
*"ama bu bir dil modeli. dil modelinin başarılı olması lazım. artık bu
bir test değil."*

### DEĞİŞMEYEN — bu kolun bütün kıyaslanabilirliği

```
graf izi     3cd9a2575e47      model_08 ile BIREBIR AYNI
olcme izi    cfafdcc15a23      model_08 ile BIREBIR AYNI
```

**Sınav değişmedi. Eğitim değişti.** Bu iki iz tutmasaydı model_08'in
sayılarıyla aynı tabloda okunamazdık.

### DEĞİŞEN — verinin temsili

```
model_08                          model_09
satir tablosu, 1 satir = 1 olgu   PAKETLENMIS METIN, pencere = 7..11 cumle
jeton semasi (varlik = 3 yuva)    KARAKTER (63 jeton)
FIM bosluk doldurma               YOK        (kullanici karari)
kisitli argmax + teacher forcing  SERBEST URETIM, TAM ESLESME
cevap yuvasi kaybi + kopru kaybi  TEK kayip: next-token
--                                BIYOGRAFI TETIKLEYICISI (yeni)
```

`"İbrahim Yılmaz hakkında ne biliyoruz?"` → varlığın olguları, her
kopyada **farklı sırada ve farklı yüzeyle**. Sınav ve kimlik
yüzeyleriyle çakışmıyor; ayrışma adın hemen ardında.

### BÜTÇE — seçilmedi, iki ölçümden türetildi

```
korpus     22.454.868 jeton      olgu basina 10 cumle/gecis (SAYILDI)
batch      32                    L4'te 226.610 yuva/s -- OLCULMUS OPTIMUM
                                 (batch buyudukce verim DUSUYOR)
adim       60.000                = 44,0 gecis -> MARUZIYET 438
                                 (model_08: 442)
esikler    isinma 6.000 (%10)  ort_bas 18.000 (%30)  olc_her 6.000
           model_08'in ORANLARI -- adim sayilari degil
sure       L4 ~1 sa 12 dk + %21 olcum  ->  ~1 sa 27 dk
```

`adım` bir tercih değil, eşit maruziyet denkleminin çözümü. Üç sayıdan
biri değişirse diğerleri de değişmeli; `ayar_09` bunu assert ediyor.

### KAPILAR — gevşetilmedi

```
SAGLIK-1HOP    one  >= 0.98
SAGLIK-EZBER   seen >= 0.95
OLGUNLUK       comp >= 0.50      altindaysa ENT YORUMLANMAZ
BIRIM TESTI    ent_yok_kisayol == 0.000
```

Yanlarında **hüküm vermeyen** bir okuma daha var: biyografi
(`recall` / `kesinlik` / `yeni_sira`). `yeni_sira`, bu kolun ana
sorusunun — **ezber mi kodlama mı** — yeni yüzeydeki hâli. Eşik
yazılmadı; bu ölçünün hiçbir kolda tabanı yok ve sayı uydurmak
önkaydın yasakladığı tahmin olurdu.

> ### ⛔ BU KOL BİR ABLASYON DEĞİLDİR
>
> model_08 ile arasındaki **hiçbir fark tek bir değişkene
> atfedilemez**: karakter tokenizer, belge biçimi, FIM'in kalkması,
> biyografi tetikleyicisi ve bütçe birlikte değişti. Dış hakemlik
> (18 Eylül) tam buna işaret etti ve haklıydı.
>
> Bu koldan çıkacak cümle *"şu değişiklik işe yaradı"* değil,
> **"bu dil, bu şekilde eğitilince şu oluyor"**. Ayırmak isteyen kol
> ayrıca koşulmalı — önkayıt §8.

### KOŞU SIRASI

Hücre 1–3 hazırlık (GPU / Drive / kod + kilit testi), 4 başlatır,
5–7 izler ve okur, 9–10 tanı. `konus_09` **taşınmadı** (hâlâ
model_08'in jeton sürümü, koşulunca durur ve sebebini söyler);
koşuya engel değil, koşu sürerken yazılacak.


In [ ]:
# 0 MODEL ADI VE YOLLAR  |  CPU  |  tekrar: GUVENLI
MODEL = "model_11"            # <-- DEGISTIRILECEK TEK SATIR

assert MODEL, ("MODEL bos. Bu SABLON -- kopyala, adini modelin adi yap "
               "(model_a.ipynb) ve bu satiri doldur.")
import time
DEPO  = "https://github.com/sekerahmet/sekerai.git"
KOD   = "/content/kod"
EV    = f"/content/drive/MyDrive/{MODEL}"     # TEPE KLASOR = MODELIN ADI
# LOG ADI BURADA URETILMEZ -- 4. hucre her BASLATMADA kendi damgasini
# basar. Sebep olculdu (15 Eylul): log adi burada uretilince, bu hucreyi
# yeniden calistirmadan ikinci bir kosu baslatmak ONCEKI kosunun logunu
# "w" ile ACIP SIFIRLIYOR. Fiilen oldu: 20.000'lik kosunun logu, 40.000'lik
# kosu baslayinca silindi. Damga BASLATAN hucrede uretilirse imkansiz.
print(MODEL, "->", EV)

In [ ]:
# 1 GPU VAR MI, BOS MU  |  GPU'yu SORAR, kullanmaz  |  tekrar: GUVENLI
import torch
assert torch.cuda.is_available(), "GPU YOK -- Calisma zamani > Turu degistir > T4"
_p = torch.cuda.get_device_properties(0)
_bos = torch.cuda.mem_get_info()[0] / 1e9
print(f"{_p.name}   toplam {_p.total_memory/1e9:.1f} GB   bos {_bos:.1f} GB")
assert _bos > 3.0, f"GPU'da sadece {_bos:.1f} GB bos -- baska bir sey mi kosuyor?"

In [ ]:
# 2 DRIVE  |  CPU  |  tekrar: GUVENLI (yalniz <model>/log/ acar)
from google.colab import drive
import os
drive.mount("/content/drive")
assert os.path.ismount("/content/drive"), \
    "drive.mount CALISMADI -- /content/drive gercek bir baglanti degil."
os.makedirs(f"{EV}/log", exist_ok=True)
with open(f"{EV}/.yazma_denemesi", "w") as f:
    f.write("ok")
os.remove(f"{EV}/.yazma_denemesi")
print("hazir ve YAZILABILIR:", EV)
print("  var olan tohum klasorleri:",
      sorted(d for d in os.listdir(EV) if d.startswith("t")) or "(yok)")

In [ ]:
# 3 KODU GITHUB'DAN CEK + KILIT TESTI  |  CPU  |  tekrar: KOSU YOKKEN (ilk isi rm -rf /content/kod)
import subprocess, os, glob, importlib, sys
subprocess.run(["rm", "-rf", KOD], check=True)
subprocess.run(["git", "clone", "--depth", "1", DEPO, KOD], check=True)
COMMIT = subprocess.run(["git", "-C", KOD, "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
print("commit:", COMMIT, "|",
      subprocess.run(["git", "-C", KOD, "log", "-1", "--format=%s"],
                     capture_output=True, text=True).stdout.strip())

# Her modelin AILE klasoru var: deneme2/model_a/{model_a.py, pencere_a.py,
# model_a.ipynb}. Klasor adini isimden TURETMIYORUZ, dosyayi ARIYORUZ --
# model_a1 gibi varyasyonlar da ayni aile klasorunde durur.
# !! model_11 ARANMAZ: kendi klasoru BELLI. Kullanici karari,
# 16 Eylul -- "model_09 diger hicbir model ile ayni seyi kullanmamali";
# model_11 ayni kurali devraliyor.
# Paylasilan sablon `deneme2/*/<MODEL>.py` glob'u yapiyordu; model_09
# ve model_11 o aramaya girmiyor.
_aday = glob.glob(f"{KOD}/deneme2/model_11/{MODEL}.py")
assert len(_aday) == 1, f"{MODEL}.py tam bir kez bulunmali, bulunan: {_aday}"
AILE = os.path.dirname(_aday[0])
_pen = glob.glob(f"{AILE}/pencere_*.py")
assert len(_pen) == 1, f"ailede tam bir pencere_*.py olmali: {_pen}"
PENCERE = _pen[0]
print("aile:", AILE, "| olcum:", os.path.basename(PENCERE))

# --- IMPORT ONBELLEGINI TEMIZLE -- BU HUCRENIN EN SESSIZ TUZAGI ----------
# `rm -rf` + yeniden klon KODU tazeler ama `sys.modules` ESKI modul
# nesnesini tutar. Ayni cekirdekte MODEL degistirip bu hucreyi yeniden
# kosarsan, yeni kodu klonlamis ama ESKI modulu kullaniyor olursun.
# OLCULDU (15 Eylul): model_a4'ten model_a5'e gecerken
#   "AssertionError: Ayar'da boyle alan yok: {'ort_bas'}"
# cikti -- cunku model_a hala onceki klondan gelen, `ort_bas`i olmayan
# nesneydi. Daha sinsi hali: alan adlari tutarsa hata VERMEZ ve kosu
# ESKI KODLA baslar; kunyedeki commit ise YENIYI gosterir.
_atilan = [n for n, m in list(sys.modules.items())
           if getattr(m, "__file__", None) and str(m.__file__).startswith(KOD)]
for n in _atilan:
    del sys.modules[n]
importlib.invalidate_caches()
if _atilan:
    print("import onbellegi temizlendi:", sorted(_atilan))

# --- KILIT: test_11.py ---------------------------------------------------
# model_11 KENDI motoruna (taban_11.py) ve KENDI verisine (veri_11.py)
# sahip -- ikisi de model_a / veri_okul KOPYASI. `test_11.py` dort sey
# tutuyor: (0) BAGIMSIZLIK, model_11/*.py disariya import ETMIYOR;
# (1) MIMARI; (2) VERI, veri_11 KENDI iddialarini tutuyor
#     (sema, soy agaci, zincir siniflari, notrluk); (3) MOTOR,
# egitim havuzu ve olcme izi model_a ile AYNI.
# Duserse egitim BASLAMAMALI -- sayilar baska bir tabloda okunur.
#
# CIKTI YAKALANIR ve BASILIR: Colab alt surec stdout'unu hucreye
# aktarmiyor; "cikti yok" ile "test kosmadi" ayrimi sansa birakilmaz.
# AYRINTI=1 -> gecen kontroller de basilir (dugme tablosu gorunur olsun).
_t = [x for x in glob.glob(f"{AILE}/test_*.py")]
if _t:
    print()
    print("=" * 72)
    _r = subprocess.run([sys.executable, _t[0]], cwd=AILE,
                        capture_output=True, text=True,
                        env={**os.environ, "AYRINTI": "1",
                             "KOSU_KOK": EV.rsplit("/", 1)[0]})
    print(_r.stdout.rstrip() or "(cikti YOK -- test kosmamis olabilir!)")
    if _r.stderr.strip():
        print("stderr:", _r.stderr.rstrip()[-2000:])
    print("=" * 72)
    assert _r.returncode == 0, (
        f"{os.path.basename(_t[0])} DUSTU (cikis {_r.returncode}). Taban "
        f"degismis olabilir ve KAYITLI sonuclar ona dayaniyor. EGITIM BASLATMA.")
    print()

sys.path.insert(0, AILE)
# !! UST KLASOR EKLENMIYOR: veri modulu de (veri_11.py) AILE icinde.
# model_11 `deneme2/` kokundeki hicbir seyi gormez.
M_11 = importlib.import_module(MODEL)
M = M_10
# `M_11.M` MOTOR (taban_11). Paylasilan sablonda `M` TABANI
# gosteriyordu (model_a); burada `M` kolun KENDISI, motor ise
# `M_11.M`. Miras denetimi taban sinifi ORADAN alir.
MOTOR = M_11.M
assert M.AYAR.ad == MODEL, f"AYAR.ad {M.AYAR.ad!r} != dosya adi {MODEL!r}"
# Yuklenen modul GERCEKTEN yeni klondan mi geldi?
assert M.__file__.startswith(KOD), f"{MODEL} {KOD} disindan geldi: {M.__file__}"
# BU KOLUN SARTLARI. Hicbiri DEVRALINMIYOR: `ayar_11.py` her alani
# `Ayar()` varsayilaninin uzerine tek tek, gerekcesiyle yaziyor.
# --- MIMARI: bu kolun TANIMI --------------------------------------
assert M.AYAR.dongu == 1,              "DONGU YOK"
assert M.AYAR.l == 8,                  "8 AYRI katman"
assert M.AYAR.dar_alfa == 0.0,         "Phi DARBOGAZI YOK"
assert M.AYAR.dar_kapi is False,       "ogrenilen GECIT YOK"
assert M.AYAR.dff == 704,              "SwiGLU d_ff = 8/3*d"
assert M.AYAR.d // M.AYAR.nh == 64,    "head_dim 64"
assert not issubclass(M_11.ModelSade, MOTOR.Model), \
    "ModelSade taban_11.Model den MIRAS ALMAMALI"
#
#   A) DILIN KENDISI (korpus)  -- hiperparametre DEGIL. "ek_kip=tr"
#      demek "bu dil ekli bir dil" demek; "bicim=3" demek "ayni olgu
#      uc yuzey biciminde geciyor" demek. Modelin secimi degil,
#      METNIN ozelligi.
#   B) SINAV BOLMELERI          -- olcumun tanimi, modele ait DEGIL.
#   C) PROJEYE OZGU VERI EKI    -- BEST PRACTICE DEGIL. Tek kalem:
#      identity bridge (ident_frac / ident_kip). model_b13'te
#      arXiv 2509.24653'ten alindi ve model_03'e kadar tasindi.
#      BURADA DURUYOR cunku recete DISI bir degisken daha katmak
#      istemiyoruz: bu kolun dugmesi VERI, identity bridge degil.
#
# Optimizasyon (wd/cosine/isinma/lr/betas) YUKARIDA degil ASAGIDA ve
# hepsi STANDART TARIF -- proje kisiti DEGIL.
# ======================================================================
# --- A) DILIN KENDISI -------------------------------------------------
# --- BU KOLUN TEK DUGMESI ------------------------------------------
assert M.AYAR.wd == 0.5,               "model_a3 RECETESI -- kolun TANIMI"
assert M.AYAR.sabit_lr is True,        "LR SABIT -- recetenin ikinci yarisi"
assert M.AYAR.veri_ad == "veri_11",    "model_11 KENDI veri modulu"
import veri_11 as _V05
assert _V05.graf_izi(_V05.kur(0)) == _V05.IZ, "graf KAYMIS"
_G05 = _V05.kur(0)
_nv, _no = sum(_G05["n"].values()), len(_G05["olgu"])
print(f"veri_11  graf izi {_V05.IZ}   {_nv} varlik  {_no} olgu  "
      f"|R| {len(_V05.ILISKI)}  {len(_V05.TIPLER)} tip"
      f"   ({_G05['n']['SEHIR']} il, TEZ ayri tip)")
assert _G05["n"]["SEHIR"] == 81,  "Turkiye'nin 81 ili"
assert "TEZ" in _V05.TIPLER,      "TEZ ayri varlik tipi"
# --- KORPUS: model_11 KARAKTER duzeyinde, PAKETLENMIS METINLE egitilir
assert M.AYAR.t_len == 512,            "egitim penceresi (Physics 3.1 Ek C)"
assert M.AYAR.kopya == 20,             "bioS multiM: varlik basina 20 belge"
assert M.AYAR.tetik * 5 == M.AYAR.kopya * 2, "biyografi onegi orani 2/5"
# --- KORPUSUN BILESIMI: hiperparametre DEGIL, METNIN ozelligi
# model_09 OLCTU: korpusun %75'i iki katli tamlama zinciriydi.
# Kullanici (CLAUDE.md kural 5): *"gercek bir metinde bu orani
# gorur muyduk"*. Zincir artik AZINLIK ve sayfalarin ICINE serpik.
assert M.AYAR.zincir_pay == 0.20,      "zincir cumlesi AZINLIK"
assert M.AYAR.n3 == 10000,             "UC adimli zincir -- zincirlerin ~%23u"
assert M.AYAR.ret_pay == 0.05,         "reddetme: kullanici, 18 Eylul '%5 uygun'"
assert 0 < M.AYAR.ret_tut < 0.5,       "reddetmenin bir kismi SINAVA ayrilir"
# EPOK: model_10 35,3 epok kosuyordu (kopya 5). model_11'de kopya 20,
# yani AYNI hesap butcesi DORT KAT daha cok benzersiz metne yayiliyor.
# Eski not: 60.000 adim ESKI korpusta 107,6 epok ederdi -- Muennighoff
# 2305.16264 44 epogu ACIKCA basarisiz rejim diye aniyor.
import ayar_11 as _AY10
print(f"butce: {M.AYAR.adim:,} adim  =  {_AY10.EPOK:.1f} epok"
      f"   korpus {_AY10.KORPUS_JETON:,} jeton")
assert M.AYAR.batch == 32,             "OLCULMUS OPTIMUM: 226.610 yuva/s"
assert M.AYAR.adim == 20000,           "CLAUDE.md kural 1 -- ILK SINIR"
# ESIKLER ORAN olarak: model_08 2000/6000/2000 @ 20.000 adim.
assert M.AYAR.isinma == M.AYAR.adim // 10,      "isinma kosunun %10u"
assert M.AYAR.ort_bas == 3 * M.AYAR.adim // 10, "ort_bas kosunun %30u"
assert M.AYAR.adim // M.AYAR.olc_her == 10,     "10 olcum noktasi"
# SATIR TABLOSU alanlari KAPALI. Korpus yolunda karsiliklari var
# (bicim -> metin_11.BICIM, soru_kat -> *_soru belgeleri, ident_frac
# -> kimlik belgeleri) ama BU ALANLAR artik hicbir yerde OKUNMUYOR.
assert M.AYAR.bicim == 1,              "satir tablosu KAPALI"
assert M.AYAR.fim_kat == 0,            "FIM cikti (kullanici, 18 Eylul)"
assert M.AYAR.soru_kat == 0,           "soru artik BELGE, satir tipi DEGIL"
assert M.AYAR.kisayol_kat == 0,        "iyelik kisa yolu satir tipiydi"
assert M.AYAR.ident_frac == 0.0,       "kimlik artik BELGE"
assert MOTOR.SPECIAL == 3,             "olu ozel jeton YOK"
assert M.AYAR.belge_pay == 0.0,        "BELGE satiri YOK (ek_kip ile kurulmadi)"

# --- B) SINAV BOLMELERI -- olcumun tanimi -----------------------------
assert M.AYAR.ood_pay == 0.05,         f"ood bolmesi: {M.AYAR.ood_pay}"
assert M.AYAR.kopru_kayip == 0.0,      "YARDIMCI KAYIP YOK (bu MIMARI karari)"

# --- C) PROJEYE OZGU VERI EKI -- BEST PRACTICE DEGIL ------------------
# Kullanici, 16 Eylul: "biz her seyi sifirdan yaptik, ben kisit
# vermedim, best practice dedim." Dogru -- ve bu IKI SATIR o tarifin
# parcasi DEGIL. Egitim havuzu b15 ile ayni kalsin diye duruyor.

# --- OPTIMIZASYON: STANDART TARIFIN KENDISI ---------------------------
# Bunlar proje kisiti DEGIL. Dordu de ayni sayilari kullaniyor:
#   nanoGPT   GPT-3   Llama   Pythia
assert M.AYAR.lr == 1e-3,              "Pythia-70m ile ayni mertebe"
assert M.AYAR.betas == (0.9, 0.95),    "nanoGPT/GPT-3/Llama/Pythia -- 0.999 DEGIL"
assert M.AYAR.tam_kayip is True,       "butun pozisyonlarda next-token: standart LM"
# --- BU KOLUN DUGMESI: GERI BESLEMELI ORTALAMA ---------------------
# !! BU KILIT GEVSETILMEDI, HEDEFI DEGISTI. model_05..model_07'de
# `ort_bas == 0` bekleniyordu ("standart tarif, projeye ozgu ne varsa
# KAPALI"). model_09'da Lookahead KOLUN DUGMESI oldu ve model_11
# onu TASIYOR, o yuzden kilit
# artik ACIK olmasini ve DOGRU DEGERDE olmasini bekliyor.
assert M.AYAR.ort_her == M.AYAR.olc_her, "her olcum bir esik"
assert M.AYAR.ort_alfa == 0.5,   "bir onceki ile YARI YARIYA"
assert M.AYAR.isinma < M.AYAR.ort_bas, "isinma BITMEDEN ortalama baslamaz"
# PROJEYE OZGU DIGER NUMARALAR -- HEPSI KAPALI
assert not (M.AYAR.dar_sert or M.AYAR.dar_sdpa), "darbogaz zaten YOK"
print("mimari: ModelSade  8 katman  RoPE  SwiGLU  bagli gomme  bias YOK")
for _g in ("AYAR", "egit", "fark_bas"):
    assert hasattr(M, _g), f"{MODEL}'de {_g} YOK -- kos_11.py duser"
print("ayar:", M.AYAR)


In [ ]:
# 4 BASLAT  |  GPU (alt surec)  |  tekrar: HAYIR -- yeni kosu baslatir
# --- GPU KAPISI (CLAUDE.md kural 2) -------------------------------------
# GPU kullanacak her hucre, ONCE kendi icinde GPU'yu sorar. Ayri bir
# "GPU var mi" hucresi HUCRE SIRASINA bagli bir kuraldir; icerideki
# kontrol degildir. Olculdu (15 Eylul): Colab cekirdegi kosu sirasinda
# oldu, GPU ve Drive durumu gitti; ayri kontrol hucresi vardi ama
# calistirilmamisti.
import torch
assert torch.cuda.is_available(), "GPU YOK -- Calisma zamani > Turu degistir > T4"
_bos = torch.cuda.mem_get_info()[0] / 1e9
assert _bos > 2.0, f"GPU'da sadece {_bos:.1f} GB bos -- baska bir sey mi kosuyor?"
print(f"GPU kapisi GECTI: {torch.cuda.get_device_name(0)}  bos {_bos:.1f} GB")
# ------------------------------------------------------------------------

TOHUMLAR = [0]             # ONCE TEK TOHUM.
# !! BU KOLUN OLCUTU `comp` VE `ent` BIRLIKTE -- onkayit model_11.md.
# Kiyas model_06'nin AYNI SINAVDAKI okumasi; ikisi de dusmediyse
# ekleme BEDELSIZ, biri dustuyse TAKAS.
# Sonuc OLUMLU cikarsa (comp yukseldiyse) tek tohum YETER.
# OLUMSUZ cikarsa bu satiri [1, 2] yapip tekrar kos -- t0 klasoru
# DOKUNULMAZ, yeni kosular t1/ ve t2/'ye yazar. Kod degismez.
# Neden onemli: grokking tohuma bagli. Tek tohumda comp yukselmezse
# "konfigurasyon yanlis" ile "bu baslangic sanssiz" AYRILAMAZ.
# VERI tohumu AYRI (ayar.veri_tohum=0) -- butun tohumlar AYNI veriyi gorur.

# !! ADIM buyutup SURDUR=True yapmak UZATMADIR, sifirdan kosu DEGIL.
#    COSINE UYARISI BU KOLDA GECERSIZ: ayar_11.sabit_lr=True, yani
#    isinmadan sonra LR SABIT. Uzatilan adimlar, bastan uzun bir
#    kosuda gorecekleri LR'nin AYNISINI gorur -- temiz uzatma.
#    (Cosine olsaydi ufuk `ayar.adim`dan turer ve LR GERI FIRLARDI:
#     model_b14'te 20.000 ufkunda lr/10 iken 60.000 ufkunda 7,6 KAT.)
ADIM   = None    # None = ayar_11.py'deki ILK SINIR (20.000).
#                  Uzatmak KULLANICI karari (CLAUDE.md kural 1).
SURDUR = False   # TAZE kosu -- surdurme DEGIL.
USTUNE = False   # t0 BOS (yeni kol). True = dolu klasoru
#                  t<N>_eski_<zaman>/'a TASI (silmez)

if "p" in globals() and p.poll() is None:
    raise SystemExit(f"ZATEN KOSUYOR (PID {p.pid}). Once 'Durdurmak' hucresi.")

# KENDI kosucusu: model_11/kos_11.py. `--model` YOK -- bu betik
# yalnizca model_11'i baslatir (paylasilan kos.py aile klasoru ARIYORDU).
_arg = [sys.executable, "-u", f"{KOD}/deneme2/model_11/kos_11.py",
        "--ev", EV, "--commit", COMMIT,
        "--tohum", *[str(t) for t in TOHUMLAR]]
if ADIM:
    _arg += ["--adim", str(ADIM)]
if SURDUR:
    _arg += ["--surdur"]
if USTUNE:
    _arg += ["--ustune"]
# LOG ADI HER BASLATMADA YENI: bu hucre iki kez calisirsa iki AYRI log
# olur, oncekinin uzerine YAZILMAZ.
LOG = f"{EV}/log/kos_{time.strftime('%Y%m%d_%H%M%S')}.txt"
p = subprocess.Popen(_arg, stdout=open(LOG, "w"), stderr=subprocess.STDOUT)
print("PID", p.pid, " tohum", TOHUMLAR, " -> log:", LOG)
print("Dolu bir tohum klasoru varsa kosu REDDEDILIR -- hicbir sey ezilmez.")
print("Ayni tohumu bilerek tekrar kosmak icin --ustune; o da SILMEZ,")
print("eskisini t<N>_eski_<zaman>/ diye yan klasore TASIR.")


In [ ]:
# 5 ILERLEME (ham log)  |  CPU  |  tekrar: GUVENLI (log geriden gelebilir)
import glob, subprocess
# LOG degiskenine DEGIL, klasordeki EN YENI log'a bak.
_l = sorted(glob.glob(f"{EV}/log/kos_*.txt"))
assert _l, f"log yok: {EV}/log/"

# `p` CEKIRDEK YENIDEN BASLAYINCA KAYBOLUR -- ve tam o an bu hucreye
# ihtiyac duyulur. Olculdu (15 Eylul): Colab cekirdegi oldu, bu hucre
# `NameError: name 'p' is not defined` verdi, koşunun yasayip yasamadigi
# ogrenilemedi. Artik `p` yoksa SUREC TABLOSUNA bakiyor.
if "p" in globals():
    _d = p.poll()
    print("KOSUYOR" if _d is None else f"BITTI/OLDU (cikis kodu {_d})",
          "| PID", p.pid)
else:
    _ps = subprocess.run(
        ["bash", "-lc", "ps -eo pid,etime,cmd | grep kos_11.py | grep -v grep"],
        capture_output=True, text=True).stdout.strip()
    print("!! `p` YOK -- cekirdek yeniden baslamis.")
    if _ps:
        print("   ama SUREC YASIYOR:\n   " + _ps)
    else:
        print("   ve kos_11.py sureci de YOK -> kosu OLDU.")
        print("   Drive'i yeniden bagla (2. hucre), 3'u kos, sonra 4. hucrede")
        print("   SURDUR = True ile KALDIGI YERDEN devam ettir.")
print("log:", _l[-1].split("/")[-1], f"({len(_l)} log dosyasi var)")
print("-" * 78)
print(open(_l[-1]).read()[-4000:])

In [ ]:
# 6 RAPOR (canli durum)  |  CPU  |  tekrar: GUVENLI
import json, glob, os, statistics

# !! BU HUCRE 18-19 Eylul'de SEKIZ KEZ DUZELTILDI. Sekizi de kullanicinin
# gozuyle bulundu; hicbirini sayisal bir kapi gostermedi.
#   1) `_yakin`/`_kisayol` sutunlarini HIC basmiyordu.
#   2) `kayip`/`bpc` icin `None`u 0.0000 diye basti -- *"hala yanlis"*.
#   3) Tablo MELEZDI: ust satirda yetenek, sutunda eski bolme adi.
#   4) *"eski olculerin turkce karsiligini yazalim demedik ki"*.
#   5) *"one ne ya?"* -- sutunlar ham JSON anahtariydi.
#   6) *"rapordan model olgun mu degil mi bilmiyorum"* -- hukum yoktu.
#   7) *"olgu aslinda ezber degil mi?"* -- `one` ve `seen` IKISI DE
#      HAFIZA; ayri alanlarmis gibi gosterilmisti.
#   8) *"kisayol dedigimiz cikarimsiz."*  kisayol AYRI BIR ARIZA GRUBU
#      DEGIL: `gorulmemis` ile AYNI bolmede, AYNI sorular uzerinde --
#      yani AYNI SORUNUN iki sonucu, DOGRUDAN TOPLANABILIRLER.
#
# AYRIM TEK SORUDA: CEVAP KORPUSTA YAZIYOR MU?
#   HAFIZA    EVET yaziyor, model BULUP GETIRECEK
#     olgu      TEK bir sey ogretildi              one
#     zincir    BILESIK bir sey ogretildi          seen
#   CIKARIM   HAYIR yazmiyor, model BIRLESTIRECEK
#     gorulmemis  o zincir korpusta HIC gecmedi    comp
#     cikarimsiz  AYNI sorular, birlestirmemis     comp_kisayol
#     yabanci     varlik hic zincir basi olmamis   ent
#   DURUSTLUK  cevapsiz soruya "yok" dedi mi       dogru_ret / kacamak
#              (model_11'den beri `egit()` ICINDE -- model_10'da
#               AYRI ARACTI ve rapor IKI PARCA kaliyordu)
#
# "OLGUN" TANIMI: CLAUDE.md "Olcu BES ALAN" -- DIL / HAFIZA / CIKARIM /
# DURUSTLUK / TUTARLILIK. Biri eksikse model olgun DEGILDIR.
#
# !! TEK HUCRE, IKI KOL. Kol numarasi `MODEL`den turer; elle yazilmaz.
_KN = MODEL.rsplit("_", 1)[1]
# !! SINAV DEGISTI, 19 Eylul. `zincir_butcesi` `v.tr2`yi budadi, yani
# `seen` LISTESI baska -> olcme izi cfafdcc15a23 -> 88f1391dfeb4.
# model_10 artik BASKA SINAV ve `KIYAS`tan `ESKI`ye TASINDI: fark
# ALINMAZ, yonelme icin durur.
# !! `comp`/`ent` LISTELERI DEGISMEDI -- CIKARIM tarafindaki kiyas
# aslinda gecerli. Ama TEK bir iz var ve o degisti; kapinin
# soyleyebilecegi tek dogru sey "sinav ayni degil".
# !! Bunlar EGRI okumasi; hukum `pencere_*` ile verilir ve bu kol
# GERI BESLEMELI -- egri ZATEN ortalanmis modeli olcuyor
# (OLCULENLER §193: model_a5'te egri +0.33, pencere -0.0177).
KIYAS = {}                    # AYNI SINAVDA kosmus kol YOK -- model_11 ILK
# BASKA SINAV -- YONELME ICIN, fark ALINMAZ.
ESKI = {
    "model_10 t0 @20.000": dict(olgu=0.9377, zincir=0.1040,
                                gorulmemis=0.0146, cikarimsiz=0.1449),
    "model_03  (veri_04)": dict(olgu=0.9857, zincir=0.9960, gorulmemis=0.8350),
    "model_07  (f4ce..)":  dict(olgu=0.9920, zincir=0.9990, gorulmemis=0.8135),
}
IZ_11 = "88f1391dfeb4"         # `test_<N>` §7c bunu gercek olcme iziyle sinar
# (json, sutun, GRUP, ne olctugu)
AD = (("wug",          "ek kurali",  "DIL",
       "UYDURMA kokte ek uyumu -- sans 0.2500"),
      ("one",          "olgu",       "HAFIZA",    "TEK bir sey ogretildi"),
      ("seen",         "zincir",     "HAFIZA",    "BILESIK bir sey ogretildi"),
      ("comp",         "gorulmemis", "CIKARIM",   "o zincir korpusta HIC gecmedi"),
      ("comp_kisayol", "cikarimsiz", "CIKARIM",
       "AYNI sorular -- birlestirmedi, tek adim yapti"),
      ("ent",          "yabanci",    "CIKARIM",   "varlik hic zincir basi olmamis"),
      ("dogru_ret",    "ret",        "DURUSTLUK", "cevapsiz soruya 'yok' dedi mi"),
      ("kacamak",      "kacamak",    "DURUSTLUK", "BILDIGINE 'yok' dedi mi -- KOTU"),
      ("ent_yok",      "birim",      "OLCUM",     "kisayol IMKANSIZ -- 0 OLMALI"),
      ("ood",          "ood",        "OLCUM",     "80 ornek -- GURULTULU"))
W = 12
_h = lambda x, n=4: (f"{'---':>{W}}" if x is None else f"{x:>{W}.{n}f}")

for kl in sorted(k for k in glob.glob(f"{EV}/t*") if os.path.isdir(k)):
    _ad = os.path.basename(kl)
    if "_eski_" in _ad:
        continue
    eg = glob.glob(f"{kl}/egri_*.json")
    if not eg:
        print(f"{_ad}: egri YOK")
        continue
    ky = glob.glob(f"{kl}/kosu_t*.json")
    k = json.load(open(ky[0])) if ky else {}
    e = json.load(open(eg[0]))
    _var = set().union(*(set(r) for r in e))
    _b = [x for x in AD if x[0] in _var]
    _yk = [c for c in _var if c.endswith("_yakin")]
    _gor = {x[0] for x in _b} | set(_yk) | {
        c for c in _var if c.endswith("_kisayol")} | {
        "adim", "kayip", "kayip_ana", "kayip_son", "bpc", "sn", "lr",
        "ret_ad", "ret_cift"}
    assert not sorted(_var - _gor), f"OLCULUYOR AMA BASILMIYOR: {sorted(_var-_gor)}"
    assert k.get("olcme_izi") == IZ_11, "BASKA SINAV"
    s, _dur = e[-1], k.get("durum", "?")
    _son = s.get("adim", 0)
    _hedef = max(_son, 20000)
    _bpc = [r["bpc"] for r in e if r.get("bpc") is not None]
    _ys = [s[c] for c in _yk if s.get(c) is not None]

    print("=" * 112)
    print(f" {_ad}   adim {_son:,}/{_hedef:,}   durum {_dur}"
          f"   commit {k.get('commit','?')}   {k.get('gpu','')}")
    print("=" * 112)
    # !! IKI NOKTA YOKSA "DUZLESTI" DENMEZ. Eski hali
    #     len(_bpc) >= 2 and ... else "DUZLESTI"
    # yaziyordu; tek noktali bir kosuda (duman testi) otomatik
    # "DUZLESTI" basiyordu -- OLMAYAN BILGIYE HUKUM. Ayni sinif
    # hata bu hucrede zaten bir kez oldu (`None` -> 0.0000).
    _eksik = []
    if len(_bpc) < 2:
        _dus, _bpc_soz = None, "TEK NOKTA -- egilim BILINMIYOR"
    else:
        _dus = _bpc[-1] < _bpc[-2] - 1e-4
        _bpc_soz = "HALA DUSUYOR -- doymadi" if _dus else "DUZLESTI"
    if _bpc:
        print(f"   {'DIL':<22}{'bpc ' + format(_bpc[-1], '.3f'):<14}"
              + _bpc_soz
              + (f"   ek hatasi {max(_ys):.4f}" if _ys else ""))
        if _dus:
            _eksik.append("bpc hala dusuyor -- daha fazla adim ISE YARAR")
        print(f"   {'':22}{'':14}!! bpc EGITIM metninde -- ezberden"
              " AYRILAMAZ (onkayit 3b/1)")
    for _g, _t, _c, _e2 in (("DIL", "ek kurali", "wug", 0.90),
                            ("HAFIZA", "olgu", "one", 0.98),
                            ("HAFIZA", "zincir", "seen", 0.95),
                            ("CIKARIM", "gorulmemis", "comp", 0.50)):
        _v = s.get(_c)
        if _v is None:
            continue
        print(f"   {_g + '/' + _t:<22}{format(_v, '.4f'):<14}"
              + ("GECTI" if _v >= _e2 else f"YETMEDI   esik {_e2}"))
        if _v < _e2:
            _eksik.append(f"{_g}/{_t} {_v:.4f} < {_e2}")
    # CIKARIM SORUSUNUN DAGILIMI -- ayni bolme, ayni sorular
    _d1, _d2 = s.get("comp"), s.get("comp_kisayol")
    if _d1 is not None and _d2 is not None:
        print(f"   {'CIKARIM/cikarimsiz':<22}{format(_d2, '.4f'):<14}"
              "birlestirmedi, TEK ADIM yapti")
        print(f"   {'':22}{'':14}-> ayni sorularin {1 - _d1 - _d2:.1%}'i"
              " NE DOGRU NE KISAYOL (tani kirar)")
    # DURUSTLUK -- CIFT olcu, tek basina OKUNMAZ
    if s.get("dogru_ret") is not None:
        print(f"   {'DURUSTLUK/ret':<22}{format(s['dogru_ret'], '.4f'):<14}"
              f"ad {s.get('ret_ad', 0):.4f}  cift {s.get('ret_cift', 0):.4f}")
        _kc = s.get("kacamak", 0.0)
        print(f"   {'DURUSTLUK/kacamak':<22}{format(_kc, '.4f'):<14}"
              + ("!! YUKSEK -- ret TEK BASINA OKUNMAZ" if _kc > 0.05
                 else "dusuk -- ret okunabilir"))
        if s["dogru_ret"] > 0.9 and _kc > 0.5:
            print("      !! MODEL HER SEYE 'YOK' DIYOR. Iki sayi da bunu soyluyor.")
    else:
        print(f"   {'DURUSTLUK':<22}{'OLCULMEDI':<14}"
              f"durustluk_{_KN} --hepsi")
    for _n2, _a2 in (("TUTARLILIK", f"konus_{_KN} --panel"),
                     ("HAFIZA/tam", f"tani_{_KN} --biyografi")):
        print(f"   {_n2:<22}{'OLCULMEDI':<14}{_a2}")
    print()
    if s.get("ent_yok_kisayol") is not None and abs(s["ent_yok_kisayol"]) >= 1e-9:
        print("   !! BIRIM TESTI DUSTU -- YUKARIDAKI SAYILARIN HICBIRI OKUNMAZ.")
    elif _dur == "KOSUYOR":
        print(f"   OLGUN MU?  KOSU DEVAM EDIYOR ({_son:,}/{_hedef:,}) -- HUKUM YOK.")
        print("      Bu kol GERI BESLEMELI: egri ZATEN ortalanmis modeli olcuyor.")
        print(f"      Hukum kosu bitince `pencere_{_KN}` ile verilir.")
    elif _eksik:
        print("   OLGUN MU?  HAYIR. Eksikler:")
        for _x in _eksik:
            print(f"      - {_x}")
    else:
        print("   OLGUN MU?  DIL / HAFIZA / CIKARIM alanlarinda EVET.")
        print("      TUTARLILIK olculmeden hukum TAMAMLANMAZ.")

    print()
    _g1, _sk = f"{'':>7}{'DIL':>{2*W}}", "DIL"
    for _a2, _t, _g, _n in _b:
        _g1 += f"{_g if _g != _sk else '':>{W}}"
        _sk = _g
    print("   " + _g1)
    print("   " + f"{'adim':>7}{'bpc':>{W}}{'ek':>{W}}"
          + "".join(f"{t:>{W}}" for _a2, t, _g, _n in _b) + f"{'dk':>6}")
    for r in e:
        _yv = [r[c] for c in _yk if r.get(c) is not None]
        print("   " + f"{r['adim']:>7d}" + _h(r.get("bpc"), 3)
              + _h(max(_yv) if _yv else None)
              + "".join(_h(r.get(a)) for a, _t, _g, _n in _b)
              + f"{r['sn']/60:>6.0f}")
    print()
    print(f"   {'sutun':<13}{'ne olculuyor':<48}json")
    print(f"   {'bpc':<13}{'karakter basina bit -- kayip/ln2':<48}bpc")
    print(f"   {'ek':<13}{'DOGRU varlik, YANLIS ek (dil bilgisi)':<48}*_yakin")
    for a, t, _g, n in _b:
        print(f"   {t:<13}{n:<48}{a}")
    d = [(b["sn"] - a["sn"]) / (b["adim"] - a["adim"]) * 1000
         for a, b in zip(e, e[1:]) if b["sn"] > a["sn"] and b["adim"] > a["adim"]]
    if d:
        print(f"\n   HIZ ortanca {statistics.median(d):.1f} ms/adim")
    if KIYAS:
        print("   KIYAS -- AYNI SINAV (iz %s), fark ALINIR." % IZ_11)
    for _n, _d in KIYAS.items():
        print(f"      {_n:<24}"
              + "  ".join(f"{a} {b:.4f}" for a, b in _d.items()))
    print("   ESKI KOLLAR -- BASKA SINAV. Yonelme icin; fark ALINMAZ.")
    for _n, _d in ESKI.items():
        print(f"      {_n:<24}"
              + "  ".join(f"{a} {b:.4f}" for a, b in _d.items()))

In [ ]:
# 7 pencere_11 -- BIRINCIL OKUMA  |  GPU  |  tekrar: GUVENLI, ama ANCAK KOSU BITINCE
# --- GPU KAPISI (CLAUDE.md kural 2) -------------------------------------
# GPU kullanacak her hucre, ONCE kendi icinde GPU'yu sorar. Ayri bir
# "GPU var mi" hucresi HUCRE SIRASINA bagli bir kuraldir; icerideki
# kontrol degildir. Olculdu (15 Eylul): Colab cekirdegi kosu sirasinda
# oldu, GPU ve Drive durumu gitti; ayri kontrol hucresi vardi ama
# calistirilmamisti.
import torch
assert torch.cuda.is_available(), "GPU YOK -- Calisma zamani > Turu degistir > T4"
_bos = torch.cuda.mem_get_info()[0] / 1e9
assert _bos > 2.0, f"GPU'da sadece {_bos:.1f} GB bos -- baska bir sey mi kosuyor?"
print(f"GPU kapisi GECTI: {torch.cuda.get_device_name(0)}  bos {_bos:.1f} GB")
# ------------------------------------------------------------------------

# BIRINCIL OKUMA. GPU kullanir (yukaridaki nota bak).
# PENCERE hucre 3'te ailenin icinden bulundu -- yolu elle yazmiyoruz.
TOHUM = TOHUMLAR[0]
!python {PENCERE} {EV}/t{TOHUM} --genislik 5

In [ ]:
# 8 DURDUR  |  CPU  |  tekrar: KOSUYU OLDURUR -- bastaki # bilerek duruyor
# Anlik goruntuler Drive'da kalir; surdurme paketi her olcum
# noktasinda yazilir, 4. hucrede SURDUR=True ile devam edilir.
# p.kill()

In [ ]:
# 9 asama1_11 --sonda -- KOPRU SONDASI  |  GPU  |  tekrar: GUVENLI
# --- GPU KAPISI (CLAUDE.md kural 2) -------------------------------------
import torch
assert torch.cuda.is_available(), "GPU YOK -- Calisma zamani > Turu degistir > T4"
_bos = torch.cuda.mem_get_info()[0] / 1e9
assert _bos > 2.0, f"GPU'da sadece {_bos:.1f} GB bos"
print(f"GPU kapisi GECTI: {torch.cuda.get_device_name(0)}  bos {_bos:.1f} GB")
# ------------------------------------------------------------------------
# ASAMA-1 TESHISI -- model_11'da kopru hidden state'e girdi mi?
# !! BU KOLUN EK OKUMASI BURADA: Physics 3.1'in iddiasi
# 'augmentation olmadan bilgi EZBERLENIR ama DOGRUSAL KODLANMAZ'.
# Sonda slot 0 'bilgi VAR' derse comp acilmasa bile bu bir bulgu.
# arXiv 2505.17923 AYNI probe'u AYNI pozisyonda yapmis ve kopruyu
# BULMUS: 'the hidden representation of the last input token
# encodes information about all necessary bridge entities'.
# model_b13'te ayni pozisyonda 0.0275 cikmisti.
# MERDIVEN: model_b6 -> b9 -> model_b10 (TABAN) -> b13 -> b8.
#   model_b9 comp 0.0250 | model_b6 0.0442 | model_b8 TAVAN 0.9997
# --sonda : kopru DOGRUSAL okunabiliyor mu. Taban max(en_sik, KOPYA) --
#           kopru cogu zaman soru varligiyla ayni aileden, soyad GIRDIDE
#           duruyor (comp'ta kopya 0.4450). Bu duzeltilmeden slot1
#           yanlislikla "BILGI VAR" cikiyordu.

import os
ASAMA1 = os.path.join(AILE, "asama1_11.py")
assert os.path.exists(ASAMA1), ASAMA1
!python {ASAMA1} {EV}/t{TOHUMLAR[0]} --bolme comp,ent,ent_yok,ood

In [ ]:
# 10 tani_11 -- ARIZA SEKLI  |  GPU  |  tekrar: GUVENLI
# --- GPU KAPISI (CLAUDE.md kural 2) -------------------------------------
import torch
assert torch.cuda.is_available(), "GPU YOK -- Calisma zamani > Turu degistir > T4"
_bos = torch.cuda.mem_get_info()[0] / 1e9
assert _bos > 2.0, f"GPU'da sadece {_bos:.1f} GB bos"
print(f"GPU kapisi GECTI: {torch.cuda.get_device_name(0)}  bos {_bos:.1f} GB")
# ------------------------------------------------------------------------
# AYRISTIRMA: model YANLIS cevap verirken NE diyor?
#   KISAYOL (r2'yi dogrudan soru varligina uygulamis)
#   KOPRU   (ara varligi yazmis, ikinci hop'u yapmamis)
#   VARLIK_DEGIL / ILGISIZ / ...
# Ayrica: 1.hop tek basina, 2.hop tek basina, IKISI BIRDEN -> KAYIP.
# Bu bir HUKUM olcusu DEGIL, arizanin SEKLINI gosterir.
import os
TANI = os.path.join(AILE, "tani_11.py")
assert os.path.exists(TANI), TANI
!python {TANI} {EV}/t{TOHUMLAR[0]}

In [ ]:
# 11 durustluk_11 -- YOK DIYEBILIYOR MU  |  GPU  |  tekrar: GUVENLI
# --- GPU KAPISI (CLAUDE.md kural 2) -------------------------------------
import torch
assert torch.cuda.is_available(), "GPU YOK -- Calisma zamani > Turu degistir"
_bos = torch.cuda.mem_get_info()[0] / 1e9
assert _bos > 2.0, f"GPU'da sadece {_bos:.1f} GB bos"
print(f"GPU kapisi GECTI: {torch.cuda.get_device_name(0)}  bos {_bos:.1f} GB")
# ------------------------------------------------------------------------
# DURUSTLUK -- CIFT olcu, ve DOKUM her zaman yaninda.
# Kullanici, 18 Eylul: *"Ama biz yok degil de olmadigini olcecegiz yani
# model olmadigini da ogrenecek direkt yok demiyecek bu da GOZLE OLCUM
# demek."*  O yuzden bu hucre sayiyi TEK BASINA basmaz.
#
#   DOGRU RET   cevapsiz soruya 'yok' dedi mi        yuksek IYI
#   KACAMAK     BILDIGI soruya 'yok' dedi mi         yuksek KOTU
#
# Sinav TUTULAN veriden: sorulan uydurma ad ve (tip, iliski) cifti
# egitimde HIC gecmiyor (ayar_11.ret_tut = 0.20). Gordugunu reddetmek
# EZBER olurdu; olctugumuz GENELLEME.
import os
DURUST = os.path.join(AILE, "durustluk_11.py")
assert os.path.exists(DURUST), DURUST
!python {DURUST} {EV}/t{TOHUMLAR[0]} --ornek 12